## Part 5: Inference function

`predict_podium(record)` answers one question: will this driver finish on the podium (top 3) in this race? It takes the pre-race features of **one driver in one race** and returns the podium probability and a yes/no answer. It uses the final model from Part 4, so nothing is retrained.

**Input contract:** `record` must be a dictionary containing all 17 engineered features from Part 3 (`FEATURES`). This function does not build those features from raw driver/race IDs. If a required feature is missing or invalid, it raises a clear error and does not make a prediction. It also rejects post-race information to help prevent data leakage.


In [ ]:
# LOCAL SETUP (dropped when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import POST_RACE_COLS
from IPython.utils.capture import capture_output
with capture_output():
    %run p4_models.ipynb

### 5.1 The function

* **Input:** a dictionary containing the 17 already-engineered features in `FEATURES` (Part 3). This function does not calculate form, standings, or circuit-history features from raw data.
* **Step 1:** check that the input is a dictionary, reject unexpected keys, and reject post-race columns (`positionOrder`, `points`, `status`, etc.).
* **Step 2:** check every required feature. A feature is **missing** if its key is absent or its value is `None` / `NaN`. A feature is **invalid** if it cannot be interpreted as a finite number or violates a basic rule for that feature (for example, a binary flag must be 0 or 1).
* **Step 3, if anything is wrong:** raise `ValueError` with a human-readable explanation of what needs fixing. No prediction is made.
* **Step 4, if all 17 features are valid:** arrange the values in the same order used for training and pass the one-row DataFrame to `FINAL_MODEL`. The model already contains the fitted preprocessing from Part 4, so preprocessing is not fitted again.
* **Output when it predicts:** a dictionary containing `status = "ok"`, the podium probability and the yes/no answer (probability at least `FINAL_THRESHOLD`).

**Why raise an error instead of filling a gap?** The Part 4 pipeline can impute some missing values with training medians, but a filled value is not the driver's actual value. This inference function follows a stricter input contract: it requires all features that should be available at the pre-race cutoff, and makes missing information visible instead of returning a potentially misleading prediction.


In [ ]:
import numpy as np
import pandas as pd

GROUP_OF = {f: g for g, cols in FEATURE_GROUPS.items() for f in cols}


def _is_missing(value):
    """Return True for None, NaN, or pandas' missing-value markers."""
    try:
        result = pd.isna(value)
        # A feature value should be scalar. Arrays/lists are handled as invalid below.
        return bool(result) if np.isscalar(result) else False
    except (TypeError, ValueError):
        return False


def _validate_feature_value(feature, value):
    """Return a readable problem description, or None when the value is acceptable."""
    if isinstance(value, (bool, np.bool_)):
        return "must be numeric, not True/False"

    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return "must be a finite number"

    if not np.isfinite(number):
        return "must be a finite number"

    # Basic domain checks based on how these features are defined in Part 3.
    binary_features = {
        "pit_lane_flag", "reached_q2", "reached_q3", "qual_format", "home_flag"
    }
    if feature in binary_features and number not in (0.0, 1.0):
        return "must be 0 or 1"

    if feature in {"driver_form5", "constructor_form5"} and not 0.0 <= number <= 1.0:
        return "must be between 0 and 1"

    if feature in {"grid", "circuit_starts_before", "circuit_podiums_before", "pts_before", "cons_pts_before"} and number < 0:
        return "cannot be negative"

    if feature in {"qual_position", "pos_before", "cons_pos_before"} and number <= 0:
        return "must be greater than 0"

    if feature == "best_qual_s" and number <= 0:
        return "must be greater than 0 seconds"

    if feature == "gap_to_pole_pct" and number < 0:
        return "cannot be negative"

    if feature in {"grid", "qual_position", "circuit_starts_before", "circuit_podiums_before"} and not number.is_integer():
        return "must be a whole number"

    return None


def predict_podium(record, model=None, threshold=None):
    """
    Predict podium probability for one driver-race record with all required
    engineered features present and valid.

    Raises:
        TypeError: if record is not a dictionary.
        ValueError: if keys/features are missing, invalid, unexpected, or leaky.
    """
    if not isinstance(record, dict):
        raise TypeError(
            "record must be a dictionary containing the 17 engineered model features. "
            f"Received {type(record).__name__} instead."
        )

    model = FINAL_MODEL if model is None else model
    threshold = FINAL_THRESHOLD if threshold is None else threshold

    # Reject post-race information before doing anything that could predict.
    leaky = set(record) & (set(POST_RACE_COLS) | {"podium"})
    if leaky:
        raise ValueError(
            "Cannot predict using post-race information. Remove these field(s): "
            + ", ".join(sorted(leaky))
        )

    # This function accepts model features only; IDs/context belong upstream.
    unexpected = set(record) - set(FEATURES)
    if unexpected:
        raise ValueError(
            "Unexpected field(s) in record: "
            + ", ".join(sorted(unexpected))
            + ". Pass only the 17 engineered features listed in FEATURES; "
              "do not pass driver/race IDs or other columns to this function."
        )

    missing = [f for f in FEATURES if f not in record or _is_missing(record[f])]
    invalid = []
    values = {}

    for feature in FEATURES:
        if feature in missing:
            continue
        problem = _validate_feature_value(feature, record[feature])
        if problem:
            invalid.append(f"{feature}: {problem}")
        else:
            values[feature] = float(record[feature])

    if missing or invalid:
        messages = ["Cannot make a podium prediction because the input record is incomplete or invalid."]
        if missing:
            grouped = [f"{feature} ({GROUP_OF.get(feature, 'unknown group')})" for feature in missing]
            messages.append(
                f"Missing {len(missing)} required feature(s): " + ", ".join(grouped) + "."
            )
        if invalid:
            messages.append("Invalid feature value(s): " + "; ".join(invalid) + ".")
        messages.append(
            f"Provide all {len(FEATURES)} required features with valid values, then call predict_podium(record) again."
        )
        raise ValueError(" ".join(messages))

    # The model receives columns in the exact training order.
    X = pd.DataFrame([[values[f] for f in FEATURES]], columns=FEATURES)
    probability = float(model.predict_proba(X)[0, 1])

    return {
        "status": "ok",
        "podium_probability": probability,
        "podium_prediction": int(probability >= threshold),
        "missing_features": [],
        "invalid_features": [],
        "message": f"Prediction made successfully: all {len(FEATURES)} required features were present and valid.",
    }


### 5.2 Example A: complete record
A real driver-race from the validation years: the first one where all features are known. The finishing position is only printed to describe the example, it is not an input.

In [ ]:
def describe_row(row):
    driver = tables["drivers"].set_index("driverId").loc[row["driverId"], "surname"]
    race = tables["races"].set_index("raceId").loc[row["raceId"], "name"]
    finish = outcomes.set_index(["raceId", "driverId"]).loc[(row["raceId"], row["driverId"]), "positionOrder"]
    return f"{driver}, {race} {row['year']} (actual finish: P{int(finish)})"

complete_rows = table[(table["split"] == "validation") & table[FEATURES].notna().all(axis=1)]
row_A = complete_rows.iloc[0]
record_A = row_A[FEATURES].to_dict()

print(describe_row(row_A))
print("input:", record_A)
print("output:", predict_podium(record_A))

All 17 features are given, so the function predicts. The output has the podium probability and the yes/no answer at the validation threshold. The function running without errors does not show that the prediction is right: one record proves nothing about accuracy. The accuracy of the model is measured in Part 4 on the validation and test sets.

### 5.3 Example B: partially missing record

This is a real validation row where some pre-race information is unavailable (for example, a driver's first-race form or standings history). Missing fields are left out of the input dictionary, as they would be if the feature record were built from the information available before the race.

Under the input contract, the function must not make a prediction for this record. It raises a `ValueError` that names the missing features and their feature groups, so the caller knows what needs to be supplied.


In [ ]:
first_race_rows = table[(table["split"] == "validation") & table["driver_form5"].isna()]
row_B = first_race_rows.iloc[0]
record_B = {k: v for k, v in row_B[FEATURES].to_dict().items() if pd.notna(v)}

print(describe_row(row_B))
print("input:", record_B)

try:
    predict_podium(record_B)
except ValueError as error:
    print("Prediction correctly refused.")
    print("Reason:", error)
else:
    raise AssertionError("The incomplete record should have raised ValueError.")


### 5.4 Checks

* For validation rows with all 17 features, the function gives the same probability as calling the final model directly.
* For validation rows with a missing feature, the function raises `ValueError` and names the missing feature(s).
* Non-numeric, non-finite, or obviously invalid feature values are rejected with an explanation.
* Post-race columns and unexpected fields are rejected before prediction.
* A non-dictionary input raises a clear `TypeError`.


In [ ]:
val = table[table["split"] == "validation"]
is_complete = val[FEATURES].notna().all(axis=1)

# Complete rows: inference wrapper must agree with the final model.
val_ok = val[is_complete]
direct = FINAL_MODEL.predict_proba(val_ok[FEATURES])[:, 1]
results_ok = [predict_podium(r[FEATURES].to_dict()) for _, r in val_ok.iterrows()]
one_by_one = np.array([r["podium_probability"] for r in results_ok])
print("complete validation rows:", len(val_ok),
      "| largest difference to the model:", abs(direct - one_by_one).max())
assert all(r["status"] == "ok" for r in results_ok)
assert np.allclose(direct, one_by_one)

# Incomplete rows: each call must raise a readable ValueError.
val_bad = val[~is_complete]
for _, row in val_bad.iterrows():
    record = {k: v for k, v in row[FEATURES].to_dict().items() if pd.notna(v)}
    try:
        predict_podium(record)
    except ValueError as error:
        expected_missing = [f for f in FEATURES if pd.isna(row[f])]
        assert all(feature in str(error) for feature in expected_missing)
    else:
        raise AssertionError("An incomplete validation record unexpectedly produced a prediction.")

print("incomplete validation rows:", len(val_bad),
      "| all raised ValueError naming the missing features")

# Invalid values, post-race fields, unexpected fields, and wrong input types.
try:
    predict_podium({**record_A, "grid": "front"})
except ValueError as error:
    assert "grid" in str(error)
    print("invalid feature value correctly refused:", error)
else:
    raise AssertionError("A non-numeric grid value should have been rejected.")

for col in ["positionOrder", "points", "status", "podium"]:
    try:
        predict_podium({**record_A, col: 1})
    except ValueError as error:
        assert col in str(error)
    else:
        raise AssertionError(f"Post-race column {col} should have been rejected.")
print("post-race leakage checks: passed")

try:
    predict_podium({**record_A, "unexpected_column": 1})
except ValueError as error:
    assert "unexpected_column" in str(error)
    print("unexpected field correctly refused:", error)
else:
    raise AssertionError("Unexpected input fields should have been rejected.")

try:
    predict_podium(None)
except TypeError as error:
    print("wrong input type correctly refused:", error)
else:
    raise AssertionError("A non-dictionary input should have been rejected.")


### 5.5 How many records cannot be predicted, and why

Because the function requires every feature, validation records with structural gaps are rejected (for example, standings at the start of a season, form for a driver's first race, or missing qualifying information). This section reports how many validation rows have at least one missing feature and which features are missing most often. The test set is not used here.


In [ ]:
share = len(val_bad) / len(val)
print(f"validation rows with missing features: {len(val_bad)} of {len(val)} ({share:.1%}); complete rows: {len(val_ok)}")

missing_counts = val[FEATURES].isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
display(
    missing_counts.rename("rows with this feature missing")
    .to_frame()
    .assign(group=lambda d: d.index.map(GROUP_OF))
)


### 5.6 Limitations

* The function takes the features computed in Part 3 (form, standings, circuit history). It does not build them from raw race data or driver/race IDs; feature construction must happen upstream.
* A record with any missing feature is rejected. This includes structural gaps (round 1 of a season, a driver's first race, or missing qualifying information), so those records cannot be predicted unless the missing features are supplied or the input policy is deliberately changed.
* It checks that features are present, numeric, finite, and within basic value rules. It cannot guarantee that the values were calculated correctly or that they accurately describe the driver and race.
* The prediction is a probability from the selected Part 4 model, not a guarantee of the actual race outcome.
